#  ASL Recognition: Hybrid CNN + Keypoints Transformer
This notebook demonstrates the **Hybrid Architecture** of our ASL Recognition system, combining:

1.  **Visual Features (CNN)**: Captures handshapes and appearance.
2.  **Structural Features (MediaPipe)**: Captures precise joint locations (Keypoints).
3.  **Temporal Modeling (Transformer)**: Fuses both feature streams to recognize words over time.

### What This Notebook Covers
1.  **The Full Inference Pipeline**: From raw video to prediction.
    *   Extracting Keypoints (MediaPipe)
    *   Extracting Visual Features (ResNet50)
    *   Feature Fusion
2.  **Model Architecture**: Loading the trained `PoseTransformer`.
3.  **Live Demo**: Visualize the pipeline on a test video.
4.  **Evaluation**: Performance metrics on the test set.


In [ ]:
import sys
import os
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from collections import defaultdict
import cv2
import mediapipe as mp
import glob
import warnings
warnings.filterwarnings('ignore')

# Local imports
PROJECT_ROOT = os.getcwd()
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.cnn_model import ASL_CNN, ASL_CNN_FeatureExtractor
from src.model import PoseTransformer
from src.dataset import CombinedFeatureDataset
from src.extract_keypoints import extract_hand_keypoints_from_image

# ═══════════════════════════════════════════════════════════════════════════
# VISUALIZATION SETUP - Beautiful dark theme
# ═══════════════════════════════════════════════════════════════════════════
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': [14, 8],
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 12,
    'figure.titlesize': 18,
    'axes.facecolor': '#1a1a2e',
    'figure.facecolor': '#0f0f1a',
    'axes.edgecolor': '#4a4a6a',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0',
    'text.color': '#e0e0e0',
    'grid.color': '#2a2a4a',
    'grid.alpha': 0.5,
})

# Custom color palette
COLORS = {
    'primary': '#6c5ce7',
    'secondary': '#00cec9',
    'accent': '#fd79a8',
    'success': '#00b894',
    'warning': '#fdcb6e',
    'danger': '#e17055',
    'info': '#74b9ff',
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")
print(f" PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")


---
## 1.  Data Loading & Exploration

We will look at the **Test Set** to pick a random video for demonstration. The system is designed to process unseen videos.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════
WORDS_ROOT = os.path.join(PROJECT_ROOT, "words")
KEYPOINTS_ROOT = os.path.join(PROJECT_ROOT, "keypoints_data")
CNN_FEATURES_ROOT = os.path.join(PROJECT_ROOT, "cnn_features")
MAX_SEQ_LEN = 60

# CNN image transform (must match training of ASL_CNN)
cnn_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ═══════════════════════════════════════════════════════════════════════════
# LOAD TRAIN / VAL / TEST DATASETS (Combined Features)
# ═══════════════════════════════════════════════════════════════════════════

# We'll mirror the training script: build the train split first to establish
# the canonical class ordering, then reuse that mapping for val/test.

datasets = {}

try:
    # Train split
    train_dataset = CombinedFeatureDataset(
        keypoints_root=KEYPOINTS_ROOT,
        cnn_features_root=CNN_FEATURES_ROOT,
        split="train",
        max_len=MAX_SEQ_LEN,
    )
    classes = train_dataset.classes
    NUM_CLASSES = len(classes)

    # Val / Test using same class mapping
    val_dataset = CombinedFeatureDataset(
        keypoints_root=KEYPOINTS_ROOT,
        cnn_features_root=CNN_FEATURES_ROOT,
        split="val",
        max_len=MAX_SEQ_LEN,
        classes=classes,
    )
    test_dataset = CombinedFeatureDataset(
        keypoints_root=KEYPOINTS_ROOT,
        cnn_features_root=CNN_FEATURES_ROOT,
        split="test",
        max_len=MAX_SEQ_LEN,
        classes=classes,
    )

    datasets["train"] = train_dataset
    datasets["val"] = val_dataset
    datasets["test"] = test_dataset

    print(f" Train samples: {len(train_dataset)}")
    print(f" Val   samples: {len(val_dataset)}")
    print(f" Test  samples: {len(test_dataset)}")
    print(f" Classes ({NUM_CLASSES}): {classes}")
except Exception as e:
    print(f"Error loading CombinedFeatureDatasets: {e}")
    print("Please ensure 'keypoints_data' and 'cnn_features' directories exist and are populated.")



In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VISUALIZE DATASET DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════
def count_samples_per_class(dataset):
    """Count how many samples we have per class name in a CombinedFeatureDataset."""
    counts = defaultdict(int)
    samples = getattr(dataset, "samples", [])
    for sample in samples:
        # CombinedFeatureDataset stores (kp_path, cnn_path, label_str)
        if len(sample) == 3:
            _, _, label = sample
        elif len(sample) == 2:
            _, label = sample
        else:
            continue
        counts[label] += 1
    return counts

if datasets:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Dataset split sizes
    split_names = list(datasets.keys())
    split_sizes = [len(datasets[s]) for s in split_names]
    colors = [COLORS['primary'], COLORS['secondary'], COLORS['accent']][:len(split_names)]

    ax1 = axes[0]
    bars = ax1.bar(split_names, split_sizes, color=colors, edgecolor='white', linewidth=2)
    ax1.set_title('Dataset Split Sizes', fontweight='bold', pad=15)
    ax1.set_ylabel('Number of Samples')
    for bar, size in zip(bars, split_sizes):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(size), ha='center', va='bottom', fontweight='bold', fontsize=14)

    # Class distribution for the train split
    ax2 = axes[1]
    train_ds = datasets.get('train')
    if train_ds is not None:
        class_counts = count_samples_per_class(train_ds)
        if class_counts:
            ax2.barh(list(class_counts.keys()), list(class_counts.values()),
                     color=COLORS['info'], edgecolor='white')
            ax2.set_title('Samples per Class (Train)', fontweight='bold', pad=15)
            ax2.set_xlabel('Count')
        else:
            ax2.text(0.5, 0.5, 'No class samples found',
                     ha='center', va='center', transform=ax2.transAxes, fontsize=14)
            ax2.set_title('Class Distribution', fontweight='bold', pad=15)
    else:
        ax2.text(0.5, 0.5, 'Train split not available',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        ax2.set_title('Class Distribution', fontweight='bold', pad=15)

    plt.tight_layout()
    plt.show()
else:
    print("No datasets available to visualize.")


###  Sample Video Sequence Visualization


In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean


def visualize_sequence(frames, label, num_frames=8):
    """Display frames from a video sequence.

    Args:
        frames: Tensor of shape (T, C, H, W) in ImageNet-normalized space.
    """
    fig, axes = plt.subplots(2, num_frames // 2, figsize=(18, 6))
    axes = axes.flatten()

    indices = np.linspace(0, frames.shape[0] - 1, num_frames, dtype=int)

    for i, ax in enumerate(axes):
        idx = indices[i]
        img = denormalize(frames[idx]).permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f'Frame {idx}', fontsize=11)
        ax.axis('off')
        # Add frame border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(COLORS['primary'])
            spine.set_linewidth(2)

    plt.suptitle(f'Video Sequence: "{label}"', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def get_random_video_frames(split="test"):
    """Load frames from a random video in words/<split>/ for visualization/demo."""
    split_dir = os.path.join(WORDS_ROOT, split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"Split directory not found: {split_dir}")

    words = [w for w in os.listdir(split_dir)
             if os.path.isdir(os.path.join(split_dir, w))]
    if not words:
        raise RuntimeError(f"No word folders found under {split_dir}")

    word = np.random.choice(words)
    word_dir = os.path.join(split_dir, word)

    video_ids = [v for v in os.listdir(word_dir)
                 if os.path.isdir(os.path.join(word_dir, v))]
    if not video_ids:
        raise RuntimeError(f"No video folders found under {word_dir}")

    video_id = np.random.choice(video_ids)
    video_dir = os.path.join(word_dir, video_id)

    frame_paths = sorted(glob.glob(os.path.join(video_dir, "frame_*.jpg")))
    frames = [Image.open(p).convert("RGB") for p in frame_paths]
    return frames, word, video_id, video_dir


# Visualize a random sample sequence from the word-level dataset
raw_frames, sample_label, sample_video_id, sample_video_dir = get_random_video_frames(split="test")
frame_tensors = torch.stack([cnn_transform(img) for img in raw_frames])  # (T, C, H, W)

print(f'Selected video: split=test, word="{sample_label}", id="{sample_video_id}"')
print(f'Number of frames: {len(raw_frames)}')
visualize_sequence(frame_tensors, sample_label)


---
## 2.  Model Architecture

Building the complete pipeline: **CNN Feature Extractor → Transformer → Classifier**


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD TRAINED MODELS
# ═══════════════════════════════════════════════════════════════════════════
CNN_WEIGHTS_PATH = os.path.join(PROJECT_ROOT, "models/best_cnn_params.pth")
PTN_WEIGHTS_PATH = os.path.join(PROJECT_ROOT, "models/best_combined_model.pth")

print(" Loading Models...")
print("-" * 50)

# 1. CNN Feature Extractor
# ------------------------
# We load the CNN trained on the alphabet to extract visual features.
# It likely has 28 or 29 output classes (A-Z...), but we only use the features.
try:
    # Check checkpoint to find num_classes
    checkpoint = torch.load(CNN_WEIGHTS_PATH, map_location=device)
    if '_base_model.fc.3.weight' in checkpoint:
        num_cnn_classes = checkpoint['_base_model.fc.3.weight'].shape[0]
    else:
        num_cnn_classes = 29 # Default guess
    
    base_cnn = ASL_CNN(num_classes=num_cnn_classes)
    base_cnn.load_state_dict(checkpoint)
    feature_extractor = ASL_CNN_FeatureExtractor(base_cnn)
    feature_extractor.to(device)
    feature_extractor.eval()
    print(f" [OK] CNN Feature Extractor loaded (Classes: {num_cnn_classes})")
except Exception as e:
    print(f" [FAIL] Could not load CNN: {e}")

# 2. Pose Transformer (The Hybrid Model)
# --------------------------------------
# Input Dim = 42 (Keypoints) + 512 (CNN Features) = 554
INPUT_DIM = 42 + 512

transformer = PoseTransformer(
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    d_model=256,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    max_len=MAX_SEQ_LEN
)

if os.path.exists(PTN_WEIGHTS_PATH):
    transformer.load_state_dict(torch.load(PTN_WEIGHTS_PATH, map_location=device))
    transformer.to(device)
    transformer.eval()
    print(f" [OK] Pose Transformer loaded from {os.path.basename(PTN_WEIGHTS_PATH)}")
else:
    print(f" [FAIL] Transformer weights not found at {PTN_WEIGHTS_PATH}")

print("-" * 50)
print(f" Hybrid Model Ready.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE PIPELINE HELPER
# ═══════════════════════════════════════════════════════════════════════════
mp_hands = mp.solutions.hands


def process_video_sequence(frames_list):
    """Full Option-B pipeline: frames → keypoints + CNN features → fused tensor.

    Args:
        frames_list: list of PIL Images (RGB).

    Returns:
        combined_features: Tensor (1, T, 554)
        keypoints_seq: Numpy (T, 42) for visualization
        cnn_features_seq: Numpy (T, 512) for visualization
    """
    # 1. Extract Keypoints (MediaPipe Hands)
    keypoints_list = []
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5) as hands:
        for img in frames_list:
            # Convert PIL to cv2 BGR
            img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            kps = extract_hand_keypoints_from_image(img_cv, hands)
            if kps is None:
                kps = np.zeros(42, dtype=np.float32)
            keypoints_list.append(kps)

    keypoints_seq = np.stack(keypoints_list)  # (T, 42)

    # 2. Extract CNN Features (pre-trained ASL_CNN)
    cnn_features_list = []
    with torch.no_grad():
        for img in frames_list:
            img_tensor = cnn_transform(img).unsqueeze(0).to(device)
            feat = feature_extractor(img_tensor)
            cnn_features_list.append(feat.cpu().numpy())

    cnn_features_seq = np.vstack(cnn_features_list)  # (T, 512)

    # 3. Combine features frame-by-frame
    # Handle potential length mismatch if any dropped frames (unlikely here)
    min_len = min(len(keypoints_seq), len(cnn_features_seq))
    combined = np.concatenate([keypoints_seq[:min_len], cnn_features_seq[:min_len]], axis=1)

    # 4. Pad/Truncate to MAX_SEQ_LEN expected by the Transformer
    T, D = combined.shape
    if T >= MAX_SEQ_LEN:
        combined_padded = combined[:MAX_SEQ_LEN, :]
    else:
        pad_len = MAX_SEQ_LEN - T
        padding = np.zeros((pad_len, D), dtype=np.float32)
        combined_padded = np.vstack([combined, padding])

    combined_tensor = torch.from_numpy(combined_padded).float().unsqueeze(0).to(device)  # (1, MaxLen, 554)

    return combined_tensor, keypoints_seq, cnn_features_seq


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LIVE DEMO: PROCESSING A SAMPLE VIDEO (END-TO-END)
# ═══════════════════════════════════════════════════════════════════════════

# 1. Select a random video from the test split (fallback to train if needed)
try:
    frames, random_word, random_video_id, video_dir = get_random_video_frames(split="test")
except Exception:
    frames, random_word, random_video_id, video_dir = get_random_video_frames(split="train")

print(f" Selected Sample: {random_word} ({random_video_id})")
print(f" Video directory: {video_dir}")
print(f" Loaded {len(frames)} frames.")

# 2. Run the full Option-B pipeline
start_time = time.time()
input_tensor, kp_seq, cnn_seq = process_video_sequence(frames)
inference_time = time.time() - start_time

# 3. Predict with the Transformer
with torch.no_grad():
    logits = transformer(input_tensor)
    probs = torch.softmax(logits, dim=1)[0]

pred_idx = probs.argmax().item()
pred_label = classes[pred_idx]
confidence = probs[pred_idx].item()

print("-" * 50)
print(f" Prediction: {pred_label.upper()} ({confidence:.2%})")
print(f" True Label: {random_word.upper()}")
print(f" Processing Time: {inference_time*1000:.1f} ms")
print("-" * 50)

# 4. Visualize: show keypoints overlaid on a middle frame
mid_idx = len(frames) // 2
mid_frame = np.array(frames[mid_idx])
mid_kps = kp_seq[mid_idx].reshape(-1, 2)

plt.figure(figsize=(10, 10))
plt.imshow(mid_frame)

# Scale keypoints to image size
h, w, _ = mid_frame.shape
if mid_kps.max() <= 1.0:
    plt.scatter(mid_kps[:, 0] * w, mid_kps[:, 1] * h, c='red', s=20, label='MediaPipe Keypoints')
else:
    plt.scatter(mid_kps[:, 0], mid_kps[:, 1], c='red', s=20, label='MediaPipe Keypoints')

plt.title(
    f"Frame {mid_idx} with Extracted Keypoints\nPred: {pred_label} ({confidence:.1%})",
    fontsize=16,
    fontweight='bold',
    color=COLORS['success'] if pred_label == random_word else COLORS['danger'],
)
plt.axis('off')
plt.legend()
plt.show()


---
## 3.  Evaluation on Test Set

We will now evaluate the model on the full test set using the pre-computed features for speed.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION LOOP
# ═══════════════════════════════════════════════════════════════════════════
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f" Evaluating on {len(test_dataset)} test samples...")

transformer.eval()
all_preds = []
all_labels = []
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = transformer(inputs)
        _, predicted = outputs.max(1)
        
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print("=" * 60)
print(f" TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 60)

# Classification Report
print("\n Classification Report:")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFUSION MATRIX
# ═══════════════════════════════════════════════════════════════════════════
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', 
            xticklabels=classes, yticklabels=classes,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('True', fontsize=14)
ax.set_title('Confusion Matrix (Test Set)', fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


---
## 4.  Comprehensive Evaluation

Full evaluation with confusion matrix and per-class metrics.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FINAL EVALUATION
# ═══════════════════════════════════════════════════════════════════════════
overall_acc = accuracy_score(all_labels, all_preds)

print("=" * 60)
print(" FINAL TEST RESULTS (Combined CNN + Keypoints Transformer)")
print("=" * 60)
print(f"  Test Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")
print("=" * 60)

print("\n Per-Class Metrics:")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# NORMALIZED CONFUSION MATRIX
# ═══════════════════════════════════════════════════════════════════════════
cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='magma',
            xticklabels=classes, yticklabels=classes,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Normalized Frequency'}, ax=ax)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('True', fontsize=14)
ax.set_title('Normalized Confusion Matrix', fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PER-CLASS ACCURACY
# ═══════════════════════════════════════════════════════════════════════════
class_correct = defaultdict(int)
class_total = defaultdict(int)

for pred, label in zip(all_preds, all_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1

class_acc = {classes[k]: class_correct[k]/class_total[k] if class_total[k] > 0 else 0 
             for k in class_total.keys()}

fig, ax = plt.subplots(figsize=(14, 6))
colors_bar = [COLORS['success'] if v > 0.8 else COLORS['warning'] if v > 0.5 else COLORS['danger'] 
              for v in class_acc.values()]

bars = ax.barh(list(class_acc.keys()), list(class_acc.values()), 
               color=colors_bar, edgecolor='white', linewidth=1.5)
ax.axvline(x=0.8, color=COLORS['success'], linestyle='--', linewidth=2, label='Target (80%)')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Accuracy', fontsize=14)
ax.set_title('Per-Class Accuracy', fontsize=18, fontweight='bold', pad=20)
ax.legend()

# Add value labels
for bar, v in zip(bars, class_acc.values()):
    ax.text(v + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{v:.2%}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 5.  Feature Space Visualization

Using t-SNE to visualize learned embeddings in 2D space.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EXTRACT TRANSFORMER EMBEDDINGS FOR VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════
transformer.eval()
all_embeddings = []
all_labels_viz = []

print(" Extracting transformer embeddings for t-SNE visualization...")

with torch.no_grad():
    for features_seq, labels in test_loader:  # features_seq: (B, T, 554)
        features_seq = features_seq.to(device)

        # Forward through the encoder stack (up to, but not including, classifier)
        x_proj = transformer.input_proj(features_seq)           # (B, T, d_model)
        x_pos = transformer.pos_encoder(x_proj)                 # (B, T, d_model)
        enc_out = transformer.transformer_encoder(x_pos)        # (B, T, d_model)

        # Global average pooling over time
        pooled = enc_out.mean(dim=1)                            # (B, d_model)

        all_embeddings.append(pooled.cpu().numpy())
        all_labels_viz.extend(labels.numpy())

all_embeddings = np.vstack(all_embeddings)
all_labels_viz = np.array(all_labels_viz)
# Backwards-compat alias so the existing t-SNE cell (which expects
# `all_features`) continues to work without modification.
all_features = all_embeddings

print(f" Extracted {len(all_embeddings)} embeddings (dim={all_embeddings.shape[1]})")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# t-SNE PROJECTION
# ═══════════════════════════════════════════════════════════════════════════
print(" Computing t-SNE projection...")

tsne = TSNE(n_components=2, perplexity=min(30, len(all_features)-1), 
            random_state=42, n_iter=1000)
features_2d = tsne.fit_transform(all_features)

print(" t-SNE complete!")

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

# Create colormap
unique_labels = np.unique(all_labels_viz)
cmap = plt.cm.get_cmap('viridis', len(unique_labels))

for i, label in enumerate(unique_labels):
    mask = all_labels_viz == label
    scatter = ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
                        c=[cmap(i)], label=classes[label], s=100, alpha=0.7,
                        edgecolors='white', linewidths=0.5)

ax.set_xlabel('t-SNE Dimension 1', fontsize=14)
ax.set_ylabel('t-SNE Dimension 2', fontsize=14)
ax.set_title('Feature Space Visualization (t-SNE)', fontsize=18, fontweight='bold', pad=20)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 6.  Temporal Attention Analysis

Analyzing how the model attends to different frames in the sequence.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ANALYZE TRANSFORMER FEATURE EVOLUTION OVER TIME
# ═══════════════════════════════════════════════════════════════════════════

if len(test_dataset) == 0:
    print("Test dataset is empty; cannot perform temporal analysis.")
else:
    sample_seq, sample_label_idx = test_dataset[0]        # (T, 554)
    sample_input = sample_seq.unsqueeze(0).to(device)      # (1, T, 554)

    with torch.no_grad():
        x_proj = transformer.input_proj(sample_input)      # (1, T, d_model)
        x_pos = transformer.pos_encoder(x_proj)
        enc_out = transformer.transformer_encoder(x_pos)   # (1, T, d_model)

    features_seq = enc_out[0].cpu().numpy()                # (T, d_model)
    T_steps, D_dim = features_seq.shape

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    # Heatmap of transformer features over time
    ax1 = axes[0]
    im = ax1.imshow(features_seq.T, aspect='auto', cmap='magma')
    ax1.set_xlabel('Time Step (Frame)', fontsize=14)
    ax1.set_ylabel('Feature Dimension', fontsize=14)
    ax1.set_title(
        f'Transformer Feature Evolution Over Time: "{classes[sample_label_idx]}"',
        fontsize=16,
        fontweight='bold',
        pad=15,
    )
    plt.colorbar(im, ax=ax1, label='Activation')

    # Feature magnitude and variance over time (attention proxy)
    ax2 = axes[1]
    feature_norms = np.linalg.norm(features_seq, axis=1)
    feature_variance = np.var(features_seq, axis=1)

    ax2.plot(range(T_steps), feature_norms, label='Feature Magnitude',
             color=COLORS['primary'], linewidth=2.5, marker='o')
    ax2.fill_between(range(T_steps), feature_norms, alpha=0.3, color=COLORS['primary'])

    ax2_twin = ax2.twinx()
    ax2_twin.plot(range(T_steps), feature_variance, label='Feature Variance',
                  color=COLORS['accent'], linewidth=2.0, linestyle='--')
    ax2_twin.set_ylabel('Variance', color=COLORS['accent'])

    ax2.set_xlabel('Time Step (Frame)', fontsize=14)
    ax2.set_ylabel('L2 Norm', fontsize=14)
    ax2.set_title('Transformer Feature Dynamics Over Sequence', fontsize=16,
                  fontweight='bold', pad=15)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


---
## 7.  Multi-Sample Inference Demo

Testing the model on multiple samples with confidence visualization.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE ON MULTIPLE TEST SAMPLES (WITH VISUALIZATION)
# ═══════════════════════════════════════════════════════════════════════════
NUM_SAMPLES = min(6, len(test_dataset))

if NUM_SAMPLES == 0:
    print("Test dataset is empty; cannot run multi-sample demo.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    transformer.eval()

    for i in range(NUM_SAMPLES):
        ax = axes[i]

        # Get fused features and label index from the CombinedFeatureDataset
        features_seq, true_label_idx = test_dataset[i]         # (T, 554), scalar
        input_tensor = features_seq.unsqueeze(0).to(device)    # (1, T, 554)

        with torch.no_grad():
            logits = transformer(input_tensor)
            probs = torch.softmax(logits, dim=1)[0]

        pred_idx = probs.argmax().item()
        pred_conf = probs[pred_idx].item()
        true_label = classes[true_label_idx]
        pred_label = classes[pred_idx]

        # Try to load the corresponding middle video frame for context
        kp_path, _, label_str = test_dataset.samples[i]
        video_id = os.path.splitext(os.path.basename(kp_path))[0]  # e.g. "about_1"
        video_dir = os.path.join(WORDS_ROOT, "test", label_str, video_id)
        frame_paths = sorted(glob.glob(os.path.join(video_dir, "frame_*.jpg")))

        if frame_paths:
            mid_frame_path = frame_paths[len(frame_paths) // 2]
            mid_frame = Image.open(mid_frame_path).convert("RGB")
            img = np.array(mid_frame) / 255.0
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No frames found", ha="center", va="center",
                    transform=ax.transAxes, fontsize=12)

        correct = pred_idx == true_label_idx
        color = COLORS['success'] if correct else COLORS['danger']

        ax.set_title(
            f'Pred: {pred_label} ({pred_conf:.1%})\nTrue: {true_label}',
            fontsize=12,
            color=color,
            fontweight='bold',
        )
        ax.axis('off')

        # Add border
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(color)
            spine.set_linewidth(3)

    plt.suptitle('Multi-Sample Inference Results (Test Set)', fontsize=18,
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIDENCE DISTRIBUTION ANALYSIS (COMBINED MODEL)
# ═══════════════════════════════════════════════════════════════════════════

if len(test_dataset) == 0:
    print("Test dataset is empty; cannot compute confidence distribution.")
else:
    transformer.eval()
    correct_confidences = []
    incorrect_confidences = []

    with torch.no_grad():
        for inputs, labels in test_loader:  # inputs: (B, T, 554)
            inputs, labels = inputs.to(device), labels.to(device)
            logits = transformer(inputs)
            probs = torch.softmax(logits, dim=1)

            confidences, predictions = probs.max(dim=1)

            for conf, pred, true in zip(confidences.cpu(), predictions.cpu(), labels.cpu()):
                if pred == true:
                    correct_confidences.append(conf.item())
                else:
                    incorrect_confidences.append(conf.item())

    fig, ax = plt.subplots(figsize=(12, 6))

    if correct_confidences:
        ax.hist(
            correct_confidences,
            bins=20,
            alpha=0.7,
            label=f'Correct ({len(correct_confidences)})',
            color=COLORS['success'],
            edgecolor='white',
            linewidth=1.5,
        )
    if incorrect_confidences:
        ax.hist(
            incorrect_confidences,
            bins=20,
            alpha=0.7,
            label=f'Incorrect ({len(incorrect_confidences)})',
            color=COLORS['danger'],
            edgecolor='white',
            linewidth=1.5,
        )

    if correct_confidences:
        ax.axvline(
            x=np.mean(correct_confidences),
            color=COLORS['success'],
            linestyle='--',
            linewidth=2,
            label='Mean (Correct)',
        )
    if incorrect_confidences:
        ax.axvline(
            x=np.mean(incorrect_confidences),
            color=COLORS['danger'],
            linestyle='--',
            linewidth=2,
            label='Mean (Incorrect)',
        )

    ax.set_xlabel('Confidence', fontsize=14)
    ax.set_ylabel('Count', fontsize=14)
    ax.set_title('Model Confidence Distribution (Test Set)', fontsize=18, fontweight='bold', pad=20)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if correct_confidences:
        print(" Confidence Statistics:")
        print(f"   Correct predictions:   μ={np.mean(correct_confidences):.3f}, σ={np.std(correct_confidences):.3f}")
    if incorrect_confidences:
        print(f"   Incorrect predictions: μ={np.mean(incorrect_confidences):.3f}, σ={np.std(incorrect_confidences):.3f}")


---
##  Summary

This notebook demonstrated the **full capabilities** of the ASL Recognition pipeline:

| Capability | Status |
|------------|--------|
| Pretrained CNN Loading |  |
| End-to-End Training |  |
| Real-time Metrics Tracking |  |
| Confusion Matrix Analysis |  |
| Per-Class Performance |  |
| t-SNE Feature Visualization |  |
| Temporal Attention Analysis |  |
| Multi-Sample Inference |  |
| Confidence Analysis |  |

### Next Steps
1. **Increase Training Data**: More diverse ASL signs
2. **Data Augmentation**: Temporal and spatial augmentations
3. **Hyperparameter Tuning**: Grid search for optimal configuration
4. **Model Ensembling**: Combine multiple models for robustness
